In [15]:
from google.colab.output import eval_js
from IPython.display import display, Javascript
from base64 import b64decode
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

In [16]:
"Model = load_model('/content/drive/MyDrive/TRAIN AI/hand_model.h5')"

"Model = load_model('/content/drive/MyDrive/TRAIN AI/hand_model.h5')"

In [17]:
LABELS = [
    "Chỉ tay tư duy, thông minh",
    "Chỉ tay trực giác và nhạy bén",
    "Chỉ tay trách nhiệm và kiên nhẫn",
    "Chỉ tay tình duyên tốt đẹp",
    "Chỉ tay tài lộc",
    "Chỉ tay sức khỏe tốt",
    "Chỉ tay sự nghiệp thành công",
    "Chỉ tay may mắn"
]

DURATIONS = {
    0: "Đường nét dài, rõ nét, ít đứt đoạn, cong nhẹ xuống lòng bàn tay (tư duy sáng tạo) hoặc chạy thẳng ngang (tư duy logic). Có xu hướng ham học hỏi và cầu tiến.",
    1: "Đường Trí đạo cong nhiều xuống gò Thái Âm kết hợp đường trực giác chạy dọc mép ngoài bàn tay. Các đường chỉ nhỏ nhưng rõ ràng, có nhiều đường phụ hướng về gò Thái Âm.",
    2: "Hai đường này xuất phát chung một đoạn. Đường Trí đạo sâu, ổn định, rất ít đường cắt ngang và đường Vận mệnh xuất hiện rõ nét.",
    3: "Đường Tâm đạo dài, liền mạch, hướng lên phía dưới ngón trỏ, không bị hình đảo hay đứt đoạn. Thường đi kèm một đường hôn nhân rõ nét.",
    4: "Có đường Thái Dương rõ ràng, đường Vận mệnh đi thẳng từ cổ tay lên. Lòng bàn tay xuất hiện hình tam giác tài lộc hoặc có hình chữ M tương đối rõ.",
    5: "Đường Sinh đạo dài và đậm, dáng ôm rộng lấy gò Kim Tinh. Đường nét liền mạch không đứt đoạn và rất ít đường cắt ngang phá vỡ.",
    6: "Đường Vận mệnh thẳng, rõ nét, chạy từ cổ tay lên giữa lòng bàn tay, ít bị gián đoạn. Có các nhánh hướng lên gò Mộc Tinh hoặc gò Thái Dương.",
    7: "Xuất hiện hình sao độc đáo ở gò Thái Dương, đường Thái Dương rõ nét, lòng bàn tay có hình chữ M và sở hữu nhiều nhánh chỉ hướng lên thay vì hướng xuống."
}


In [20]:
def capture_hand_camera(filename='hand_captured.jpg'):
  js = Javascript('''
    async function takePhoto() {
      const div = document.createElement('div');
      div.style.padding = '15px';
      div.style.background = '#1e1e24';
      div.style.color = '#ffffff';
      div.style.width = '240px';
      div.style.textAlign = 'center';
      div.style.fontFamily = 'Arial, sans-serif';
      div.style.borderRadius = '10px';
      div.style.margin = '10px auto';

      const title = document.createElement('h3');
      title.textContent = "AI HAND SCANNER";
      title.style.margin = '0 0 10px 0';
      title.style.fontSize = '14px';
      div.appendChild(title);

      const video = document.createElement('video');
      video.style.display = 'block';
      video.style.width = '200px';
      video.style.height = '200px';
      video.style.margin = '0 auto';
      video.style.borderRadius = '6px';
      video.style.backgroundColor = '#000';
      div.appendChild(video);

      const capture = document.createElement('button');
      capture.textContent = 'BÓI CHỈ TAY AI';
      capture.style.marginTop = '12px';
      capture.style.padding = '10px 20px';
      capture.style.background = '#007bff';
      capture.style.color = 'white';
      capture.style.border = 'none';
      capture.style.borderRadius = '5px';
      capture.style.fontWeight = 'bold';
      capture.style.cursor = 'pointer';
      capture.style.width = '100%';
      div.appendChild(capture);

      const stream = await navigator.mediaDevices.getUserMedia({video: {width: 200, height: 200}});
      document.body.appendChild(div);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = 200;
      canvas.height = 200;
      canvas.getContext('2d').drawImage(video, 0, 0, 200, 200);

      stream.getTracks().forEach(track => track.stop());

      const imgPreview = document.createElement('img');
      imgPreview.src = canvas.toDataURL('image/jpeg', 0.85);
      imgPreview.style.width = '200px';
      imgPreview.style.height = '200px';
      imgPreview.style.borderRadius = '6px';
      imgPreview.style.margin = '0 auto';
      imgPreview.style.display = 'block';

      video.remove();
      capture.remove();
      div.insertBefore(imgPreview, div.firstChild.nextSibling); // Chèn ảnh vào dưới tiêu đề

      return canvas.toDataURL('image/jpeg', 0.85);
    }
    ''')
  display(js)
  data = eval_js('takePhoto()')
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

In [21]:
try:
    print("Mới bạn đưa lòng bàn tay trước Webcam laptop và ấn nút...")
    img_path = capture_hand_camera()

    img = cv2.imread(img_path)
    if img is None:
        raise ValueError("Không thể đọc được file ảnh đã chụp.")

    img_resized = cv2.resize(img, (100, 100), interpolation=cv2.INTER_AREA)
    img_normalized = img_resized / 255.0
    img_input = np.expand_dims(img_normalized, axis=0)

    print("\n" + "="*55)
    print("Hệ thống đã phân tích xong!")
    print("MODEL PREDICTION OUTPUT:")
    print("="*55)

    if model is not None:
        raw_predictions = model.predict(img_input)
        predictions = np.squeeze(raw_predictions)

        if len(predictions.shape) == 0:
            predictions = np.array([predictions])

        THRESHOLD = 0.4 # Ngưỡng lọc (40%)
        detected_any = False
        sorted_indices = np.argsort(predictions)[::-1]

        count = 0
        for idx in sorted_indices:
            if idx >= len(LABELS):
                continue

            conf = predictions[idx]
            if conf >= THRESHOLD and count < 2:
                print(f" {count+1}: {LABELS[idx]} — Độ chính xác: {conf*100:.2f}%")
                print(f" {DURATIONS[idx]}\n")
                detected_any = True
                count += 1

        if not detected_any and len(sorted_indices) > 0:
            highest_idx = sorted_indices[0]
            if highest_idx < len(LABELS):
                print(f" {LABELS[highest_idx]} — Độ chính xác: {predictions[highest_idx]*100:.2f}%")
                print(f" {DURATIONS[highest_idx]}\n")

    print("="*55)

except Exception as err:
    print(f" Hệ thống gặp lỗi xử lý: {err}")


Mới bạn đưa lòng bàn tay trước Webcam laptop và ấn nút...


<IPython.core.display.Javascript object>


Hệ thống đã phân tích xong!
MODEL PREDICTION OUTPUT:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
 Chỉ tay trách nhiệm và kiên nhẫn — Độ chính xác: 17.11%
 Hai đường này xuất phát chung một đoạn. Đường Trí đạo sâu, ổn định, rất ít đường cắt ngang và đường Vận mệnh xuất hiện rõ nét.

